In [79]:
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

In [80]:
df = pd.read_csv('../dataset/dataset1.csv')
df.drop(columns=['Unnamed: 0'], inplace=True)
df

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.7150,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.2670,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.1200,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.1430,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.1670,119.949,4,acoustic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113995,2C3TZjDRiAzdyViavDJ217,Rainy Lullaby,#mindfulness - Soft Rain for Mindful Meditatio...,Sleep My Little Boy,21,384999,False,0.172,0.2350,5,-16.393,1,0.0422,0.6400,0.928000,0.0863,0.0339,125.995,5,world-music
113996,1hIz5L4IB9hN3WRYPOCGPw,Rainy Lullaby,#mindfulness - Soft Rain for Mindful Meditatio...,Water Into Light,22,385000,False,0.174,0.1170,0,-18.318,0,0.0401,0.9940,0.976000,0.1050,0.0350,85.239,4,world-music
113997,6x8ZfSoqDjuNa5SVP5QjvX,Cesária Evora,Best Of,Miss Perfumado,22,271466,False,0.629,0.3290,0,-10.895,0,0.0420,0.8670,0.000000,0.0839,0.7430,132.378,4,world-music
113998,2e6sXL2bYv4bSz6VTdnfLs,Michael W. Smith,Change Your World,Friends,41,283893,False,0.587,0.5060,7,-10.889,1,0.0297,0.3810,0.000000,0.2700,0.4130,135.960,4,world-music


In [81]:
df.info()
df.dropna(inplace=True)
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   track_id          114000 non-null  str    
 1   artists           113999 non-null  str    
 2   album_name        113999 non-null  str    
 3   track_name        113999 non-null  str    
 4   popularity        114000 non-null  int64  
 5   duration_ms       114000 non-null  int64  
 6   explicit          114000 non-null  bool   
 7   danceability      114000 non-null  float64
 8   energy            114000 non-null  float64
 9   key               114000 non-null  int64  
 10  loudness          114000 non-null  float64
 11  mode              114000 non-null  int64  
 12  speechiness       114000 non-null  float64
 13  acousticness      114000 non-null  float64
 14  instrumentalness  114000 non-null  float64
 15  liveness          114000 non-null  float64
 16  valence           114000 non-nu

track_id            0
artists             0
album_name          0
track_name          0
popularity          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
track_genre         0
dtype: int64

In [82]:
df.columns = df.columns.str.upper()
df['EXPLICIT'] = df['EXPLICIT'].astype(int)  # True/False -> 1/0
df['DURATION_MIN'] = df['DURATION_MS'] / 60000
df.drop_duplicates(subset=['TRACK_ID'], inplace=True)
df.drop_duplicates(subset=['TRACK_NAME'], inplace=True)
df.reset_index(drop=True, inplace=True)
df

,TRACK_ID,ARTISTS,ALBUM_NAME,TRACK_NAME,POPULARITY,DURATION_MS,EXPLICIT,DANCEABILITY,ENERGY,KEY,...,MODE,SPEECHINESS,ACOUSTICNESS,INSTRUMENTALNESS,LIVENESS,VALENCE,TEMPO,TIME_SIGNATURE,TRACK_GENRE,DURATION_MIN
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,0,0.676,0.4610,1,...,0,0.1430,0.0322,0.000001,0.3580,0.7150,87.917,4,acoustic,3.844433
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,0,0.420,0.1660,1,...,1,0.0763,0.9240,0.000006,0.1010,0.2670,77.489,4,acoustic,2.493500
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,0,0.438,0.3590,0,...,1,0.0557,0.2100,0.000000,0.1170,0.1200,76.332,4,acoustic,3.513767
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,0,0.266,0.0596,0,...,1,0.0363,0.9050,0.000071,0.1320,0.1430,181.740,3,acoustic,3.365550
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,0,0.618,0.4430,2,...,1,0.0526,0.4690,0.000000,0.0829,0.1670,119.949,4,acoustic,3.314217
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73603,4WbOUe6T0sozC7z5ZJgiAA,Lucas Cervetti,Frecuencias Álmicas en 432hz,"Frecuencia Álmica, Pt. 4",22,305454,0,0.331,0.1710,1,...,1,0.0350,0.9200,0.022900,0.0679,0.3270,132.147,3,world-music,5.090900
73604,2C3TZjDRiAzdyViavDJ217,Rainy Lullaby,#mindfulness - Soft Rain for Mindful Meditatio...,Sleep My Little Boy,21,384999,0,0.172,0.2350,5,...,1,0.0422,0.6400,0.928000,0.0863,0.0339,125.995,5,world-music,6.416650
73605,1hIz5L4IB9hN3WRYPOCGPw,Rainy Lullaby,#mindfulness - Soft Rain for Mindful Meditatio...,Water Into Light,22,385000,0,0.174,0.1170,0,...,0,0.0401,0.9940,0.976000,0.1050,0.0350,85.239,4,world-music,6.416667
73606,6x8ZfSoqDjuNa5SVP5QjvX,Cesária Evora,Best Of,Miss Perfumado,22,271466,0,0.629,0.3290,0,...,0,0.0420,0.8670,0.000000,0.0839,0.7430,132.378,4,world-music,4.524433


In [83]:
features = [
    'DANCEABILITY', 'ENERGY', 'LOUDNESS', 'SPEECHINESS',
    'ACOUSTICNESS', 'INSTRUMENTALNESS', 'LIVENESS',
    'VALENCE', 'TEMPO'
]
weights = np.array([1.2, 1.5, 0.8, 1.0, 1.0, 1.0, 1.0, 1.3, 0.7])

In [84]:
scaler = StandardScaler()
scaled_features = scaler.fit_transform(df[features]) * weights
scaled_features

array([[ 0.78863374, -1.02023105,  0.27944933, ...,  0.67221997,
         1.21981316, -0.79412371],
       [-0.93226725, -2.73081989, -1.29792513, ..., -0.60454765,
        -0.98567917, -1.03607105],
       [-0.8112664 , -1.61168889, -0.16989712, ..., -0.52506017,
        -1.70935634, -1.06291542],
       ...,
       [-2.58594554, -3.0149516 , -1.46079066, ..., -0.58467578,
        -2.12780912, -0.85625786],
       [ 0.47268707, -1.78564707, -0.34449257, ..., -0.68949989,
         1.35765643,  0.23744713],
       [-0.21970669, -0.86946728, -0.24057745, ..., -0.66267287,
         1.18535234, -0.99641934]], shape=(73608, 9))

In [85]:
model = NearestNeighbors(metric='cosine', algorithm='brute')
model.fit(scaled_features)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None


In [87]:
distances, indices = model.kneighbors([scaled_features[49727]], n_neighbors=20)
print(distances)
print(indices)
# x = df.index[df['ARTISTS'] == 'Artcell']
# print(x)
# df.iloc[x]
df.iloc[indices[0]]

[[1.11022302e-16 5.58990319e-03 5.99900482e-03 8.35364927e-03
  9.51324738e-03 1.22598831e-02 1.25996270e-02 1.30248313e-02
  1.42883102e-02 1.54632933e-02 1.57364467e-02 1.65083479e-02
  1.68026771e-02 1.68337755e-02 1.71347777e-02 1.76262255e-02
  1.78726279e-02 1.83273675e-02 1.83861457e-02 1.84114960e-02]]
[[49727 41921 31109 37241 37387 37067 58516 37471 35136 15975  9291 58184
  30481 37009 70687 30975 30965 10601 25260 30758]]


,TRACK_ID,ARTISTS,ALBUM_NAME,TRACK_NAME,POPULARITY,DURATION_MS,EXPLICIT,DANCEABILITY,ENERGY,KEY,...,MODE,SPEECHINESS,ACOUSTICNESS,INSTRUMENTALNESS,LIVENESS,VALENCE,TEMPO,TIME_SIGNATURE,TRACK_GENRE,DURATION_MIN
49727,4X6kRZnZM8mZPkeMUmn8G6,Artcell,Oniket Prantor,Aniket Prantor,28,980079,0,0.500,0.923,10,...,1,0.0888,0.037400,0.003540,0.1140,0.500,137.998,4,metal,16.334650
41921,5vOxXwnRerzYPSZ8Lts55j,Sevendust,Dying To Live,Dying to Live,23,189105,0,0.498,0.948,6,...,1,0.0776,0.000072,0.034800,0.0775,0.509,134.978,4,industrial,3.151750
31109,0HngfIkijdET7l6e0IMqJ6,Ghoultown,Ghost of the Southern Son,Southern Son,21,197597,0,0.482,0.953,6,...,1,0.0778,0.000246,0.007790,0.0701,0.522,136.966,4,goth,3.293283
37241,21QHgIyqsupJivqr7AIxa1,Malon,Serie De Oro,Gatillo Facil,27,177986,0,0.477,0.970,11,...,0,0.1090,0.000023,0.001840,0.0960,0.487,131.518,4,heavy-metal,2.966433
37387,2WAauBcy5xb2spT2TKrTXN,Malon,Espíritu Combativo,Gatillo Fácil,24,176581,0,0.474,0.982,11,...,0,0.1110,0.000030,0.004790,0.1010,0.489,131.561,4,heavy-metal,2.943017
37067,7iOSeHfKU2EhSKcX7TbjKG,Beast In Black,Berserker,Blind and Frozen,61,304596,0,0.491,0.950,2,...,1,0.0590,0.000032,0.000017,0.0698,0.497,146.007,4,heavy-metal,5.076600
58516,7tCPyTEXUH1UU1JQLzd9zu,The Runaways,On air 70's Hits,Cherry Bomb,0,137640,0,0.495,0.855,11,...,1,0.0694,0.042700,0.000041,0.1410,0.496,136.735,4,punk,2.294000
37471,6PXwTSUvMWBkW6qeCtr8OD,Accept,The Rise of Chaos,Carry the Weight,23,274800,0,0.498,0.994,4,...,0,0.0664,0.000205,0.037900,0.1090,0.504,131.013,4,heavy-metal,4.580000
35136,05RgAMGypEvqhNs5hPCbMS,Van Halen,1984 (Remastered),Panama - 2015 Remaster,74,210226,0,0.526,0.978,8,...,1,0.1080,0.001220,0.000048,0.0744,0.451,141.167,4,hard-rock,3.503767
15975,6BcKHm3M9k108jKf2C8h1l,Cole Swindell;Lainey Wilson,Easy Country,Never Say Never,0,177173,0,0.469,0.931,0,...,0,0.0563,0.007520,0.000000,0.1110,0.490,140.059,4,country,2.952883
